In [ ]:
import numpy as np 
import networkx as nx 
import matplotlib.pyplot as plt

oG = nx.karate_club_graph()

def modularity(G):
    A = nx.to_numpy_array(G, weight=None)
    K = A.sum(axis=1)
    m = A.sum()/2
    return A - np.outer(K,K)/(2*m) , m

B, m = modularity(oG)
c = ["red","blue","green","purple","orange","yellow"]*10

metrics = ["Degree","Betweenness","Closeness","Clustering"]
hist = {name: {n: [] for n in oG.nodes()} for name in metrics}

def history(G, depth, vals):
    for name, d in vals.items():
        for n, v in d.items():
            hist[name][n].append((depth, v))

def display(G, depth=0):
    vals = {
        "Degree": nx.degree_centrality(G),
        "Betweenness": nx.betweenness_centrality(G),
        "Closeness": nx.closeness_centrality(G),
        "Clustering": nx.clustering(G),
    }
    print(f"Depth {depth}, nodes {list(G.nodes())}")
    print(f"Degree Centrality = {vals['Degree']}")
    print(f"Betweenness Centrality = {vals['Betweenness']}")
    print(f"Closeness Centrality = {vals['Closeness']}")
    print(f"Clustering Coefficient = {vals['Clustering']}")
    history(G, depth, vals)
    nx.draw_spring(G, with_labels=True, node_color=c[0])
    c.remove(c[0])
    plt.show()

def recursion(G, depth=0):
    display(G, depth)
    if len(G)<=1:
        return [list(G.nodes())]
    index = list(G.nodes())
    Blocal = B[np.ix_(index,index)].copy()
    Blocal -= np.diag(Blocal.sum(axis=1))
    values, vecmat = np.linalg.eigh(Blocal)
    lam_da = max(values)
    v = vecmat[:, -1]
    s = np.where(v>=0, 1, -1)
    if lam_da <= 1e-10 or s@Blocal@s/(4*m) <= 1e-10:
        return [list(G.nodes())]
    comp1 = []
    comp2 = []
    for node,s_ in zip(index, s):
        if s_ == 1:
            comp1.append(node)
        else:
            comp2.append(node)   
    if not comp1 or not comp2:
        return [list(G.nodes())]
    return recursion(G.subgraph(comp1), depth+1)+recursion(G.subgraph(comp2), depth+1)

communities = recursion(oG)
print(communities)

# evolution of each metric per node
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, name in zip(axes.ravel(), metrics):
    mean_val = {n: np.mean([v for _, v in pts]) for n, pts in hist[name].items()}
    top = sorted(mean_val, key=mean_val.get, reverse=True)[:5]      # highlight 5 strongest nodes
    for n, pts in hist[name].items():
        d, v = zip(*pts)
        if n in top:
            ax.plot(d, v, marker="o", linewidth=2, label=f"node {n}")
        else:
            ax.plot(d, v, marker=".", linewidth=0.8, color="lightgray")
    ax.set_title(f"{name} across split depth")
    ax.set_xlabel("Split depth (iteration)")
    ax.set_ylabel(name)
    ax.set_xticks(range(max(len(p) for p in hist[name].values())))
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

#Summary
for name in metrics:
    mean_val = {n: np.mean([v for _, v in pts]) for n, pts in hist[name].items()}
    top = sorted(mean_val, key=mean_val.get, reverse=True)[:5]
    print(f"{name}: top 5 by mean across iterations -> {top}")
